# last_test_a — reg_lgbm 전체 HPO + 다양성 axes (PC1, ~18시간)

**목적**: 11-base stacking plateau (val=0.005701) 깰 수 있는 신규 다양성 base를 reg_lgbm 아키텍처에서 탐색.

**탐색 axes (14축)**:
- LGBM HP 11종 (n_estimators, lr, num_leaves, max_depth, min_child_samples, subsample, colsample_bytree, reg_alpha, reg_lambda, min_split_gain, path_smooth)
- `objective` ∈ {regression, poisson, tweedie_1.2, 1.3, 1.5, 1.7, 1.8} — 7종 (확장)
- `target_transform` ∈ {log1p, none, sqrt} — 3종 (확장)
- `feat_subset` ∈ {full, top_300, top_200, top_100} — 4종 (확장)

**알고리즘**: Optuna TPE (n_startup_trials=30 → 첫 30개는 랜덤 탐색으로 다양성 확보, 이후 TPE 최적화)

**예산**: 18시간 timeout. 평균 trial ~3-5분 → ~250-360 trials 예상.

**산출**: 모든 trial이 `4_output/last_test_a/trials/trial_NNNN/`에 oof/val/test_unit.csv + meta.json 저장. 끝에서 `summary.csv` 자동 생성.

**게이트 (사후 검사)**: 단독 val < 0.005994 AND max_corr_with_11base < 0.97.

**실행 방법**: 이 노트북만 다른 PC로 옮겨서 실행하면 됨 (Colab/Local 자동 부트스트랩).

## 1. 환경 + import (self-contained)

In [8]:
import os, sys, json, time, math

GDRIVE_FINAL_ID = '1HR7LlQmp4n9wGh2WneyVex2mCZ-poiY9'

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/final/modules/preprocess.py'):
        assert GDRIVE_FINAL_ID, 'GDRIVE_FINAL_ID 비어있음'
        os.makedirs('/content/project/3_modeling/final', exist_ok=True)
        os.system(f'gdown {GDRIVE_FINAL_ID} -O /content/final.zip')
        os.system('unzip -qo /content/final.zip -d /content/project/3_modeling/final')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess

import lightgbm as lgb
import optuna
from sklearn.model_selection import KFold

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'SEED = {SEED}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
SEED = 42


## 2. 설정

In [9]:
USER       = 'jh'
N_FOLDS    = 5
CLIP_Y_EXTREME = True

TIMEOUT_HOURS = 18.0
TIMEOUT_SEC   = int(TIMEOUT_HOURS * 3600)
N_TRIALS_CAP  = 600
N_STARTUP     = 30

OUT_DIR = os.path.join(OUTPUT_DIR, 'last_test_a')
os.makedirs(OUT_DIR, exist_ok=True)

EXP_ID = 'last-test-a-reg-lgbm-hpo'
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')

BEST_SINGLE_VAL = 0.005709
THR_SINGLE_VAL  = BEST_SINGLE_VAL * 1.05
THR_CORR_VS_11  = 0.97

print(f'EXP_ID={EXP_ID}')
print(f'TIMEOUT={TIMEOUT_HOURS:.1f}h ({TIMEOUT_SEC}s) | N_TRIALS_CAP={N_TRIALS_CAP} | N_STARTUP={N_STARTUP}')
print(f'OUT_DIR={OUT_DIR}')
print(f'게이트: single_val<{THR_SINGLE_VAL:.6f}, max_corr_vs_11<{THR_CORR_VS_11}')
print(f'저장 정책: 매 trial 메타는 SQLite의 user_attrs에 누적, best_*.csv 1세트만 덮어쓰기')

EXP_ID=last-test-a-reg-lgbm-hpo
TIMEOUT=18.0h (64800s) | N_TRIALS_CAP=600 | N_STARTUP=30
OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\last_test_a
게이트: single_val<0.005994, max_corr_vs_11<0.97
저장 정책: 매 trial 메타는 SQLite의 user_attrs에 누적, best_*.csv 1세트만 덮어쓰기


## 3. 데이터 로드 + 전처리 (1회)

In [10]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = int((y_raw >= 1.0).sum())
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip ({n_clipped}개)')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

# 03b log1p preset 전처리 1회
PARAMS = {
    'missing_threshold':          0.5,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.25,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.99,
    'post_impute_corr_keep_by':   'std',
}
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values

y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit).values.astype(np.float64)
assert not pd.isna(y_train_die_broadcast).any(), 'unmapped train die y'
y_bin_die_broadcast = (y_train_die_broadcast > 0).astype(np.int32)

unit_ids_train_unique = y_train_unit.index.values
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(unit_ids_train_unique))

n_train_die = len(X_train_die)
n_val_die   = len(X_val_die)
n_test_die  = len(X_test_die)

print(f'\n[cleaning] feat_cols_clean={len(feat_cols_clean)}')
print(f'  X_train_die: {X_train_die.shape}, val: {X_val_die.shape}, test: {X_test_die.shape}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')
print(f'  fold: {N_FOLDS}-fold unit-level shuffle SEED={SEED}')

Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip (1개)
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=50%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 4개 컬럼 추가 (결측률 >= 25%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, dist<=5.0): 156,772개 채움 → 잔여: 186,722
  2단계 (lot 평균, train 기준): 105,526개 채움 → 잔여: 81,196
  3단계 (train 전체 평균): 81,196개 채움 → 잔여: 0

  [요약] 343,494 → 공간(156,772) → lot(105,526) → 전체(81,196) → 잔여(0)

[고상관 제거] threshold=0.99, keep_by=std (std)
  제거: 0개, 잔여: 564개
  

## 4. Feature gain importance 캐싱 (top_300/200/100)

In [11]:
import pickle

with open(os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'lgbm', 'fold_models.pkl'), 'rb') as f:
    fm_data = pickle.load(f)

# fold_models.pkl 구조 호환:
#   (a) dict {'fold_models': [...], 'feature_names': [...], ...} — 표준
#   (b) dict인데 'feature_names'가 None — 과거 저장본
#   (c) list [LGBMRegressor, ...] — legacy
if isinstance(fm_data, dict):
    fold_models_list = fm_data.get('fold_models', None)
    lgbm_feat_names  = fm_data.get('feature_names', None)
else:
    fold_models_list = list(fm_data)
    lgbm_feat_names  = None

assert fold_models_list is not None and len(fold_models_list) > 0, 'fold_models.pkl 비어있음'

# feature_names가 None이면 booster에서 직접 복구
def _recover_feat_names(models):
    m0 = models[0]
    if hasattr(m0, 'booster_'):
        names = m0.booster_.feature_name()
    elif hasattr(m0, 'feature_name_'):
        names = list(m0.feature_name_)
    else:
        raise RuntimeError('cached LGBM model에서 feature_name 복구 실패')
    return list(names)

if lgbm_feat_names is None or not hasattr(lgbm_feat_names, '__iter__'):
    print('[fold_models.pkl] feature_names=None → booster에서 복구')
    lgbm_feat_names = _recover_feat_names(fold_models_list)
else:
    lgbm_feat_names = list(lgbm_feat_names)

print(f'fold_models: {len(fold_models_list)}개 | feature_names: {len(lgbm_feat_names)}개')

gains = []
for m in fold_models_list:
    if hasattr(m, 'booster_'):
        gains.append(m.booster_.feature_importance(importance_type='gain'))
    elif hasattr(m, 'feature_importances_'):
        gains.append(m.feature_importances_)
mean_gain = np.mean(gains, axis=0)
assert len(mean_gain) == len(lgbm_feat_names), \
    f'gain 길이({len(mean_gain)}) != feat_names 길이({len(lgbm_feat_names)})'

common_feats = [f for f in lgbm_feat_names if f in feat_cols_clean]
print(f'  교집합 (lgbm_feat_names ∩ feat_cols_clean): {len(common_feats)}개')
assert len(common_feats) >= 100, f'common_feats가 너무 적음({len(common_feats)}) — 전처리 PARAMS 미스매치 의심'

gain_dict = {f: g for f, g in zip(lgbm_feat_names, mean_gain)}
common_gains = np.array([gain_dict[f] for f in common_feats])

# top_300, top_200, top_100
def _top_k_idx(k):
    k_eff = min(k, len(common_feats))
    sorted_feats = [f for f, g in sorted(zip(common_feats, common_gains), key=lambda x: -x[1])[:k_eff]]
    return [feat_cols_clean.index(f) for f in sorted_feats]

TOP_300_IDX = _top_k_idx(300)
TOP_200_IDX = _top_k_idx(200)
TOP_100_IDX = _top_k_idx(100)

FEAT_SUBSET_MAP = {
    'full':    None,
    'top_300': TOP_300_IDX,
    'top_200': TOP_200_IDX,
    'top_100': TOP_100_IDX,
}
print(f'feat subset 캐싱: full({len(feat_cols_clean)}), '
      f'top_300({len(TOP_300_IDX)}), top_200({len(TOP_200_IDX)}), top_100({len(TOP_100_IDX)})')

fold_models: 5개 | feature_names: 568개
  교집합 (lgbm_feat_names ∩ feat_cols_clean): 568개
feat subset 캐싱: full(568), top_300(300), top_200(200), top_100(100)


## 5. 11-base OOF 잔차 로드 (게이트 corr 계산용)

In [12]:
BASE_OOF_PATHS = {
    'zit_only':                 os.path.join(OUTPUT_DIR, 'final', 'zit_only',      'oof_unit.csv'),
    'bag_zit_combined_best':    os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_combined_best',    'oof_unit.csv'),
    'bag_zit_hpo':              os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_hpo',              'oof_unit.csv'),
    'bag_zit_combined_best_xy': os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_combined_best_xy', 'oof_unit.csv'),
    'bag_zit_pp_hpo':           os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_pp_hpo',           'oof_unit.csv'),
    'bag_zit_fixed_ge':         os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_fixed_ge',         'oof_unit.csv'),
    'reg__catboost':            os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'catboost', 'oof_unit.csv'),
    'reg__lgbm':                os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'lgbm',     'oof_unit.csv'),
    'reg__et':                  os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'et',       'oof_unit.csv'),
    'reg__enet':                os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'enet',     'oof_unit.csv'),
}

def _load_base_pred(path, y_series):
    df = pd.read_csv(path)
    return df.set_index(KEY_COL)['pred'].reindex(y_series.index).values

base_oof_pred = {k: _load_base_pred(p, y_train_unit) for k, p in BASE_OOF_PATHS.items()}
base_oof_resid = {k: y_train_unit.values - v for k, v in base_oof_pred.items()}
print(f'[11-base OOF 잔차 로드] {len(base_oof_resid)}개')

[11-base OOF 잔차 로드] 10개


## 6. 헬퍼 함수

In [13]:
def _mean_die_to_unit(pred_die, uid_die):
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    pred_sum = np.zeros(len(unique_units))
    cnt      = np.zeros(len(unique_units))
    np.add.at(pred_sum, inverse, pred_die)
    np.add.at(cnt,      inverse, 1.0)
    return pred_sum / cnt, unique_units


def _rmse_unit(pred_unit_arr, unique_units, y_unit_series):
    s = pd.Series(pred_unit_arr, index=unique_units).reindex(y_unit_series.index)
    return float(np.sqrt(np.mean((s.values - y_unit_series.values) ** 2)))


def _save_unit_csv(uids, pred, y_true, path):
    pd.DataFrame({KEY_COL: uids, 'pred': pred, TARGET_COL: y_true}).to_csv(path, index=False)


def _max_corr_with_11base(oof_unit_pred_aligned):
    new_resid = y_train_unit.values - oof_unit_pred_aligned
    # 모델이 상수로 붕괴(분산 0) → corr 정의 불가 → 게이트 fail (1.0)로 처리
    if np.std(new_resid) < 1e-12:
        return 1.0
    cs = []
    for r in base_oof_resid.values():
        c = float(np.corrcoef(new_resid, r)[0, 1])
        if not np.isfinite(c):
            c = 1.0
        cs.append(c)
    return max(cs)


def _slice_features(feat_subset_name):
    idx = FEAT_SUBSET_MAP[feat_subset_name]
    if idx is None:
        return X_train_die, X_val_die, X_test_die, len(feat_cols_clean)
    return X_train_die[:, idx], X_val_die[:, idx], X_test_die[:, idx], len(idx)


# ── 글로벌 best 트래커 (val_rmse 기준) ──
BEST_VAL_RMSE = float('inf')

def _maybe_save_best(oof_s, val_s, test_s, val_rmse, meta_dict):
    """새 best (val_rmse 기준)이면 best_*.csv + best_meta.json 덮어쓰기."""
    global BEST_VAL_RMSE
    if val_rmse < BEST_VAL_RMSE:
        BEST_VAL_RMSE = val_rmse
        _save_unit_csv(y_train_unit.index.values, oof_s.values,  y_train_unit.values, os.path.join(OUT_DIR, 'best_oof_unit.csv'))
        _save_unit_csv(y_val_unit.index.values,   val_s.values,  y_val_unit.values,   os.path.join(OUT_DIR, 'best_val_unit.csv'))
        _save_unit_csv(y_test_unit.index.values,  test_s.values, y_test_unit.values,  os.path.join(OUT_DIR, 'best_test_unit.csv'))
        with open(os.path.join(OUT_DIR, 'best_meta.json'), 'w', encoding='utf-8') as f:
            json.dump(meta_dict, f, indent=2, ensure_ascii=False, default=str)
        return True
    return False


def train_reg_lgbm_trial(hp_full, target_transform, feat_subset_name):
    """학습 + 예측 반환 (저장 안 함). 호출자가 best 갱신 여부 판단."""
    Xtr_full, Xvl_full, Xte_full, n_feat_used = _slice_features(feat_subset_name)

    oof_die_pred  = np.full(n_train_die, np.nan)
    val_die_pred  = np.zeros(n_val_die)
    test_die_pred = np.zeros(n_test_die)

    t0 = time.time()
    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unit_ids_train_unique[tr_uidx]
        vl_units = unit_ids_train_unique[vl_uidx]
        tr_die_mask = np.isin(uid_train_die, tr_units)
        vl_die_mask = np.isin(uid_train_die, vl_units)
        X_tr = Xtr_full[tr_die_mask]
        X_vl = Xtr_full[vl_die_mask]
        y_tr = y_train_die_broadcast[tr_die_mask]

        if target_transform == 'log1p':
            y_tr_fit = np.log1p(y_tr)
        elif target_transform == 'none':
            y_tr_fit = y_tr
        elif target_transform == 'sqrt':
            y_tr_fit = np.sqrt(np.maximum(y_tr, 0.0))
        else:
            raise ValueError(target_transform)

        m = lgb.LGBMRegressor(**hp_full)
        m.fit(X_tr, y_tr_fit)

        for X_pred, target_arr in [
            (X_vl,    oof_die_pred),
            (Xvl_full, val_die_pred),
            (Xte_full, test_die_pred),
        ]:
            p = m.predict(X_pred)
            if target_transform == 'log1p':
                p = np.clip(np.expm1(p), 0.0, None)
            elif target_transform == 'sqrt':
                p = np.clip(p, 0.0, None) ** 2
            else:
                p = np.clip(p, 0.0, None)
            if X_pred is X_vl:
                target_arr[vl_die_mask] = p
            else:
                target_arr += p / N_FOLDS

    assert not np.isnan(oof_die_pred).any()

    oof_u, oof_uids = _mean_die_to_unit(oof_die_pred, uid_train_die)
    val_u, val_uids = _mean_die_to_unit(val_die_pred, uid_val_die)
    test_u, test_uids = _mean_die_to_unit(test_die_pred, uid_test_die)

    oof_rmse  = _rmse_unit(oof_u,  oof_uids,  y_train_unit)
    val_rmse  = _rmse_unit(val_u,  val_uids,  y_val_unit)
    test_rmse = _rmse_unit(test_u, test_uids, y_test_unit)

    oof_s  = pd.Series(oof_u,  index=oof_uids).reindex(y_train_unit.index)
    val_s  = pd.Series(val_u,  index=val_uids).reindex(y_val_unit.index)
    test_s = pd.Series(test_u, index=test_uids).reindex(y_test_unit.index)

    max_corr11 = _max_corr_with_11base(oof_s.values)

    return {
        'oof_rmse': oof_rmse, 'val_rmse': val_rmse, 'test_rmse': test_rmse,
        'max_corr_vs_11base': max_corr11,
        'oof_s': oof_s, 'val_s': val_s, 'test_s': test_s,
        'n_feat_used': n_feat_used,
        'elapsed': time.time() - t0,
    }


print('train_reg_lgbm_trial 정의 완료')

train_reg_lgbm_trial 정의 완료


## 7. Optuna HPO 실행 (TIMEOUT_SEC 자동 종료)

In [14]:
def objective_a(trial):
    hp = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 3000),
        learning_rate=trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        num_leaves=trial.suggest_int('num_leaves', 8, 512),
        max_depth=trial.suggest_int('max_depth', 3, 14),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 400),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.1, 1.0),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-8, 30.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        min_split_gain=trial.suggest_float('min_split_gain', 1e-9, 1.0, log=True),
        path_smooth=trial.suggest_float('path_smooth', 0.0, 50.0),
        random_state=SEED, n_jobs=7, verbose=-1,  # 14코어 PC에서 둘 동시 실행 → 절반 분배
    )
    obj_choice = trial.suggest_categorical(
        'objective',
        ['regression', 'poisson', 'tweedie_1.2', 'tweedie_1.3',
         'tweedie_1.5', 'tweedie_1.7', 'tweedie_1.8'],
    )
    if obj_choice == 'poisson':
        hp['objective'] = 'poisson'
    elif obj_choice.startswith('tweedie'):
        hp['objective'] = 'tweedie'
        hp['tweedie_variance_power'] = float(obj_choice.split('_')[1])
    else:
        hp['objective'] = 'regression'

    target_transform = trial.suggest_categorical('target_transform', ['log1p', 'none', 'sqrt'])
    feat_subset      = trial.suggest_categorical('feat_subset',      ['full', 'top_300', 'top_200', 'top_100'])

    r = train_reg_lgbm_trial(hp, target_transform, feat_subset)

    # SQLite user_attrs에 trial 메타 누적
    trial.set_user_attr('val_rmse',             r['val_rmse'])
    trial.set_user_attr('test_rmse',            r['test_rmse'])
    trial.set_user_attr('max_corr_vs_11base',   r['max_corr_vs_11base'])
    trial.set_user_attr('n_feat_used',          r['n_feat_used'])
    trial.set_user_attr('elapsed_s',            r['elapsed'])
    trial.set_user_attr('hp_objective_resolved', hp['objective'])
    if 'tweedie_variance_power' in hp:
        trial.set_user_attr('tweedie_variance_power', hp['tweedie_variance_power'])

    # best 갱신 시에만 CSV 덮어쓰기
    meta_dict = {
        'trial_number': trial.number,
        'oof_rmse': r['oof_rmse'], 'val_rmse': r['val_rmse'], 'test_rmse': r['test_rmse'],
        'max_corr_vs_11base': r['max_corr_vs_11base'],
        'hp_full': {k: (float(v) if hasattr(v, 'item') else v) for k, v in hp.items()},
        'target_transform': target_transform,
        'feat_subset': feat_subset,
        'n_features_used': r['n_feat_used'],
        'CLIP_Y_EXTREME': CLIP_Y_EXTREME,
        'feat_cols_clean_n': len(feat_cols_clean),
        'SEED': int(SEED),
        'elapsed_seconds': r['elapsed'],
    }
    is_new_best = _maybe_save_best(r['oof_s'], r['val_s'], r['test_s'], r['val_rmse'], meta_dict)
    best_mark = ' ★new_best' if is_new_best else ''

    p1 = '✅' if r['val_rmse'] < THR_SINGLE_VAL else '❌'
    p2 = '✅' if r['max_corr_vs_11base'] < THR_CORR_VS_11 else '❌'
    print(f'trial {trial.number:04d} | obj={obj_choice} tt={target_transform} feat={feat_subset} '
          f'| val={r["val_rmse"]:.6f}{p1} corr={r["max_corr_vs_11base"]:.4f}{p2} '
          f'| {r["elapsed"]:.0f}s{best_mark}')

    return r['oof_rmse']


study = optuna.create_study(
    direction='minimize',
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=N_STARTUP),
)
study.set_user_attr('exp_memo', 'last_test_a — reg_lgbm full HPO + diversity (objective × target_transform × feat_subset)')
study.set_user_attr('timeout_hours', TIMEOUT_HOURS)

# 재개 시 BEST_VAL_RMSE 복구 (DB에서 읽음)
for t in study.trials:
    if t.state == optuna.trial.TrialState.COMPLETE:
        v = t.user_attrs.get('val_rmse', float('inf'))
        if v < BEST_VAL_RMSE:
            BEST_VAL_RMSE = v
print(f'\n현재 best_val_rmse (복구): {BEST_VAL_RMSE if BEST_VAL_RMSE != float("inf") else "없음"}')

print(f'\n=== Optuna HPO 시작 (timeout={TIMEOUT_HOURS}h, startup={N_STARTUP}, sampler=TPE) ===')
print(f'  탐색 axes: 11 LGBM HP + objective(7) + target_transform(3) + feat_subset(4) = 14축')
t_total = time.time()
try:
    study.optimize(objective_a, n_trials=N_TRIALS_CAP, timeout=TIMEOUT_SEC)
except KeyboardInterrupt:
    print('\n[KeyboardInterrupt] HPO 중단 — 지금까지 trial은 SQLite에 저장됨')

print(f'\n[HPO 종료] {time.time()-t_total:.0f}s, completed trials: {len(study.trials)}')
print(f'best OOF RMSE = {study.best_value:.6f}')
print(f'best params:')
for k, v in study.best_trial.params.items():
    print(f'  {k:25s} = {v}')


현재 best_val_rmse (복구): 0.005750628529176247

=== Optuna HPO 시작 (timeout=18.0h, startup=30, sampler=TPE) ===
  탐색 axes: 11 LGBM HP + objective(7) + target_transform(3) + feat_subset(4) = 14축
trial 0005 | obj=regression tt=sqrt feat=top_100 | val=0.006000❌ corr=0.9958❌ | 226s
trial 0006 | obj=tweedie_1.2 tt=log1p feat=top_200 | val=0.005751✅ corr=0.9979❌ | 238s
trial 0007 | obj=poisson tt=none feat=top_200 | val=0.005941✅ corr=0.9952❌ | 368s
trial 0008 | obj=tweedie_1.2 tt=none feat=full | val=0.005970✅ corr=0.9952❌ | 453s
trial 0009 | obj=poisson tt=log1p feat=full | val=0.005735✅ corr=0.9989❌ | 230s ★new_best
trial 0010 | obj=tweedie_1.3 tt=log1p feat=full | val=0.005741✅ corr=0.9954❌ | 232s
trial 0011 | obj=tweedie_1.5 tt=sqrt feat=top_100 | val=0.006072❌ corr=0.9958❌ | 278s
trial 0012 | obj=tweedie_1.8 tt=log1p feat=top_300 | val=0.006106❌ corr=0.9961❌ | 251s
trial 0013 | obj=tweedie_1.8 tt=none feat=full | val=0.005797✅ corr=0.9980❌ | 386s
trial 0014 | obj=tweedie_1.7 tt=sqrt feat=

## 8. 전체 trial summary + 게이트 통과 후보 출력

In [15]:
# SQLite의 모든 trial info 추출 → summary.csv
rows = []
for t in study.trials:
    if t.state != optuna.trial.TrialState.COMPLETE:
        continue
    row = {
        'trial': t.number,
        'oof_rmse':        t.value,                              # objective return (= oof_rmse)
        'val_rmse':        t.user_attrs.get('val_rmse'),
        'test_rmse':       t.user_attrs.get('test_rmse'),
        'max_corr_11':     t.user_attrs.get('max_corr_vs_11base'),
        'n_feat_used':     t.user_attrs.get('n_feat_used'),
        'elapsed_s':       t.user_attrs.get('elapsed_s'),
        # axes
        **{f'param_{k}': v for k, v in t.params.items()},
    }
    row['pass_1'] = (row['val_rmse'] is not None) and (row['val_rmse'] < THR_SINGLE_VAL)
    row['pass_2'] = (row['max_corr_11'] is not None) and (row['max_corr_11'] < THR_CORR_VS_11)
    row['pre_pass'] = row['pass_1'] and row['pass_2']
    rows.append(row)

if len(rows) == 0:
    print('=== 완료된 trial 없음. SQLite에 FAIL 또는 RUNNING 상태만 있음 ===')
    n_fail = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.FAIL)
    n_run  = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.RUNNING)
    print(f'  FAIL: {n_fail} | RUNNING: {n_run} | total: {len(study.trials)}')
    summary_df = pd.DataFrame()
else:
    summary_df = pd.DataFrame(rows).sort_values('val_rmse')
    summary_df.to_csv(os.path.join(OUT_DIR, 'summary.csv'), index=False)

    print(f'=== 전체 complete trial: {len(summary_df)} ===')
    print(f'  게이트 1 (val<{THR_SINGLE_VAL:.6f}) 통과: {summary_df["pass_1"].sum()}')
    print(f'  게이트 2 (corr<{THR_CORR_VS_11}) 통과: {summary_df["pass_2"].sum()}')
    print(f'  둘 다 통과 (pre_pass): {summary_df["pre_pass"].sum()}')

    key_cols = ['trial','val_rmse','test_rmse','max_corr_11','pass_1','pass_2','pre_pass',
                'param_objective','param_target_transform','param_reg_objective','param_feat_subset','param_w0','param_clf_scale_pos_weight']
    display_cols = [c for c in key_cols if c in summary_df.columns]

    print(f'\n=== 상위 10 (val_rmse 기준) ===')
    print(summary_df.head(10)[display_cols].to_string(index=False))

    pass_df = summary_df[summary_df['pre_pass']]
    print(f'\n=== 게이트 통과 후보: {len(pass_df)}개 (있으면 best_*.csv 갱신과 별개 — 다양성 후보로 가치) ===')
    if len(pass_df) > 0:
        print(pass_df.head(30)[display_cols].to_string(index=False))
        print(f'\n⚠ 통과 trial이 best_val_rmse 아닌 경우, 그 trial의 OOF는 SQLite엔 메타만 있고 CSV 없음.')
        print(f'   필요 시 해당 HP로 재실행해서 CSV 받기. (요청 정책상 best 1개만 보관)')

# best_*.csv 파일 확인
print(f'\n=== best_*.csv 상태 (val_rmse={BEST_VAL_RMSE:.6f}) ===')
for fn in ['best_oof_unit.csv', 'best_val_unit.csv', 'best_test_unit.csv', 'best_meta.json']:
    p = os.path.join(OUT_DIR, fn)
    if os.path.exists(p):
        sz = os.path.getsize(p) / 1024
        print(f'  {fn:25s}  {sz:>8.1f} KB')
    else:
        print(f'  {fn:25s}  (없음)')

# Colab → 로컬 자동 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', os.path.basename(OUT_DIR) + '_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'\n[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024/1024:.1f} MB)')
    try:
        files.download(_zip_path)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip_path))
except ImportError:
    pass

=== 전체 complete trial: 233 ===
  게이트 1 (val<0.005994) 통과: 215
  게이트 2 (corr<0.97) 통과: 1
  둘 다 통과 (pre_pass): 0

=== 상위 10 (val_rmse 기준) ===
 trial  val_rmse  test_rmse  max_corr_11  pass_1  pass_2  pre_pass param_objective param_target_transform param_feat_subset
   212  0.005727   0.008427     0.999114    True   False     False     tweedie_1.3                   none              full
   173  0.005727   0.008426     0.999197    True   False     False     tweedie_1.3                   none           top_200
   199  0.005727   0.008427     0.999119    True   False     False     tweedie_1.3                   none              full
   202  0.005727   0.008427     0.998948    True   False     False     tweedie_1.3                   none              full
   192  0.005728   0.008428     0.999137    True   False     False     tweedie_1.3                   none              full
   226  0.005728   0.008426     0.998941    True   False     False     tweedie_1.3                   none           